# TE02_004 - Command to Motion Direction Validation

This notebook validates TE02_004 using live ROS 2 subscriptions while a rosbag is played in a terminal.

For this KPI, `/telejoy` is treated as the low-level velocity command sent to the machine. The validation checks whether the measured machine feedback in `/joints` moves in the expected direction after each command.

## Workflow

1. Start your ROS 2 environment.
2. Start this notebook and run the acquisition cell.
3. In a terminal, play the bag.

```bash
ros2 bag play /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_001/result/WBag/bag_20260707_121849
```

If your bag publishes `/clock` and the rest of the system uses simulated time, set `USE_SIM_TIME = True` in the configuration cell before acquisition.

## Method

The notebook records `/telejoy`, `/joints`, `/encoders`, `/joint_states`, and `/statusjoy`. It then detects command windows where one `/telejoy` channel is active, measures the feedback change before and after each command window, and checks whether the sign of the feedback movement matches the expected direction.

The physical command mapping is intentionally configurable in `ACTIONS_TO_VALIDATE`, because the bag only contains the six-element command vector and not the semantic name of each joystick axis/button. Update that table once with the official controller mapping, then rerun the analysis cells.

In [ ]:
from pathlib import Path
import time
import os
import numpy as np
import pandas as pd
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
from std_msgs.msg import Float32MultiArray, Int8MultiArray, UInt8

KPI_DIR = Path(str(os.environ.get('HOME')) + '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004')
KPI_DIR.mkdir(parents=True, exist_ok=True)

RAW_COMMAND_CSV = KPI_DIR / 'te02_004_direction_raw_telejoy.csv'
RAW_JOINTS_CSV = KPI_DIR / 'te02_004_direction_raw_joints.csv'
RAW_ENCODERS_CSV = KPI_DIR / 'te02_004_direction_raw_encoders.csv'
RAW_JOINT_STATES_CSV = KPI_DIR / 'te02_004_direction_raw_joint_states.csv'
EVENTS_CSV = KPI_DIR / 'te02_004_direction_events.csv'
RESULTS_CSV = KPI_DIR / 'te02_004_direction_results.csv'
SUMMARY_CSV = KPI_DIR / 'te02_004_direction_summary.csv'

TELEJOY_TOPIC = '/telejoy'
JOINTS_TOPIC = '/joints'
ENCODERS_TOPIC = '/encoders'
JOINT_STATES_TOPIC = '/joint_states'
STATUS_TOPIC = '/statusjoy'

COMMAND_VECTOR_LENGTH = 6
COMMAND_DEADBAND = 10
MIN_EVENT_DURATION_S = 0.25
PRE_WINDOW_S = 0.20
POST_WINDOW_S = 0.70
MIN_ABS_FEEDBACK_DELTA = 1e-4
ACQUISITION_DURATION_S = 170.0
USE_SIM_TIME = False

# Raw /joints layout from telehandler_moveit/scripts/joint_remap.py:
# /joints[0] = arm raise/lower angle in degrees, mapped to cylinder_to_arm1_joint with a negative sign.
# /joints[1] = boom extension length in metres, mapped to arm prismatic joints.
# /joints[2] = cabin/base rotation angle in degrees, mapped to base_to_cylinder_joint.
# /joints[3] = fork/end angle in degrees.
# /joints[4:] = optional cage/winch signals when available.

# Edit this mapping with the official remote-controller semantics.
# command_sign_to_feedback_sign defines what should happen when telejoy[command_index] is positive.
# Use +1 if the feedback signal should increase, -1 if it should decrease.
ACTIONS_TO_VALIDATE = [
    # Example mappings. Verify these against the official remote before using the result in the final report.
    {'action': 'cabin_rotate_positive', 'command_index': 0, 'feedback_source': 'joints', 'feedback_index': 2, 'command_sign_to_feedback_sign': +1},
    {'action': 'arm_raise_positive', 'command_index': 1, 'feedback_source': 'joints', 'feedback_index': 0, 'command_sign_to_feedback_sign': +1},
    {'action': 'boom_extend_positive', 'command_index': 2, 'feedback_source': 'joints', 'feedback_index': 1, 'command_sign_to_feedback_sign': +1},
]

print(f'Output directory: {KPI_DIR}')
print('Configured actions:')
display(pd.DataFrame(ACTIONS_TO_VALIDATE))

Output directory: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004
Configured actions:


,action,command_index,feedback_source,feedback_index,command_sign_to_feedback_sign
0,cabin_rotate_positive,0,joints,2,1
1,arm_raise_positive,1,joints,0,1
2,boom_extend_positive,2,joints,1,1


In [2]:
class TelejoyMotionLogger(Node):
    def __init__(self):
        super().__init__('te02_004_telejoy_motion_logger')
        if USE_SIM_TIME:
            self.set_parameters([rclpy.parameter.Parameter('use_sim_time', rclpy.Parameter.Type.BOOL, True)])

        self.telejoy_rows = []
        self.joints_rows = []
        self.encoders_rows = []
        self.joint_state_rows = []
        self.status_rows = []

        self.create_subscription(Int8MultiArray, TELEJOY_TOPIC, self.telejoy_cb, 10)
        self.create_subscription(Float32MultiArray, JOINTS_TOPIC, self.joints_cb, 10)
        self.create_subscription(Float32MultiArray, ENCODERS_TOPIC, self.encoders_cb, 10)
        self.create_subscription(JointState, JOINT_STATES_TOPIC, self.joint_state_cb, 10)
        self.create_subscription(UInt8, STATUS_TOPIC, self.status_cb, 10)
        self.create_timer(5.0, self.health_cb)

    def now_ns(self):
        return self.get_clock().now().nanoseconds

    def append_vector(self, rows, values):
        now_ns = self.now_ns()
        rows.append({'time_ns': now_ns, 'time_s': now_ns / 1e9, 'values': list(values)})

    def telejoy_cb(self, msg):
        now_ns = self.now_ns()
        values = [int(v) for v in msg.data]
        row = {'time_ns': now_ns, 'time_s': now_ns / 1e9, 'values': values}
        for i in range(COMMAND_VECTOR_LENGTH):
            row[f'cmd_{i}'] = values[i] if i < len(values) else np.nan
        self.telejoy_rows.append(row)

    def joints_cb(self, msg):
        self.append_vector(self.joints_rows, msg.data)

    def encoders_cb(self, msg):
        self.append_vector(self.encoders_rows, msg.data)

    def joint_state_cb(self, msg):
        now_ns = self.now_ns()
        row = {'time_ns': now_ns, 'time_s': now_ns / 1e9}
        for name, position in zip(msg.name, msg.position):
            row[name] = position
        self.joint_state_rows.append(row)

    def status_cb(self, msg):
        now_ns = self.now_ns()
        self.status_rows.append({'time_ns': now_ns, 'time_s': now_ns / 1e9, 'status': int(msg.data)})

    def health_cb(self):
        self.get_logger().info(
            f'samples: telejoy={len(self.telejoy_rows)}, joints={len(self.joints_rows)}, '
            f'encoders={len(self.encoders_rows)}, joint_states={len(self.joint_state_rows)}, '
            f'status={len(self.status_rows)}'
        )

In [3]:
if not rclpy.ok():
    rclpy.init()

logger = TelejoyMotionLogger()
deadline = time.monotonic() + ACQUISITION_DURATION_S
print(f'Acquiring for {ACQUISITION_DURATION_S:.1f} s. Play the bag now if it is not already running.')

try:
    while time.monotonic() < deadline:
        rclpy.spin_once(logger, timeout_sec=0.1)
except KeyboardInterrupt:
    print('Acquisition interrupted by user.')
finally:
    telejoy_df = pd.DataFrame(logger.telejoy_rows)
    joints_df = pd.DataFrame(logger.joints_rows)
    encoders_df = pd.DataFrame(logger.encoders_rows)
    joint_states_df = pd.DataFrame(logger.joint_state_rows)
    status_df = pd.DataFrame(logger.status_rows)
    logger.destroy_node()

telejoy_df.to_csv(RAW_COMMAND_CSV, index=False)
joints_df.to_csv(RAW_JOINTS_CSV, index=False)
encoders_df.to_csv(RAW_ENCODERS_CSV, index=False)
joint_states_df.to_csv(RAW_JOINT_STATES_CSV, index=False)

print(f'telejoy samples: {len(telejoy_df)}')
print(f'joints samples: {len(joints_df)}')
print(f'encoders samples: {len(encoders_df)}')
print(f'joint_states samples: {len(joint_states_df)}')
display(telejoy_df.head())
display(joints_df.head())

Acquiring for 170.0 s. Play the bag now if it is not already running.


[INFO] [1783434898.966462312] [te02_004_telejoy_motion_logger]: samples: telejoy=0, joints=26, encoders=441, joint_states=26, status=441
[INFO] [1783434903.941069142] [te02_004_telejoy_motion_logger]: samples: telejoy=0, joints=76, encoders=1304, joint_states=76, status=1304
[INFO] [1783434908.940644006] [te02_004_telejoy_motion_logger]: samples: telejoy=0, joints=126, encoders=2111, joint_states=126, status=2112
[INFO] [1783434913.940789655] [te02_004_telejoy_motion_logger]: samples: telejoy=0, joints=176, encoders=2861, joint_states=176, status=2862
[INFO] [1783434918.940798164] [te02_004_telejoy_motion_logger]: samples: telejoy=0, joints=226, encoders=3701, joint_states=226, status=3702
[INFO] [1783434923.940899486] [te02_004_telejoy_motion_logger]: samples: telejoy=0, joints=276, encoders=4597, joint_states=276, status=4597
[INFO] [1783434928.940695641] [te02_004_telejoy_motion_logger]: samples: telejoy=0, joints=326, encoders=5520, joint_states=326, status=5521
[INFO] [1783434933.

telejoy samples: 0
joints samples: 1616
encoders samples: 27080
joint_states samples: 1616


""


,time_ns,time_s,values
0,1783434896404433647,1.783435e+09,"[1.8999998569488525, 6.390909194946289, -0.499..."
1,1783434896504069240,1.783435e+09,"[1.8999998569488525, 6.390909194946289, -0.499..."
2,1783434896605080263,1.783435e+09,"[1.8999998569488525, 6.390909194946289, -0.499..."
3,1783434896703658785,1.783435e+09,"[1.8999998569488525, 6.390909194946289, -0.499..."
4,1783434896805793766,1.783435e+09,"[1.8999998569488525, 6.390909194946289, -0.499..."


In [4]:
def expand_vector_df(df, prefix):
    if df.empty:
        return pd.DataFrame(columns=['time_ns', 'time_s'])
    rows = []
    for _, row in df.iterrows():
        out = {'time_ns': row['time_ns'], 'time_s': row['time_s']}
        values = row['values']
        for i, value in enumerate(values):
            out[f'{prefix}_{i}'] = float(value)
        rows.append(out)
    return pd.DataFrame(rows).sort_values('time_ns').reset_index(drop=True)

joints_expanded = expand_vector_df(joints_df, 'joints')
encoders_expanded = expand_vector_df(encoders_df, 'encoders')
joint_states_expanded = joint_states_df.sort_values('time_ns').reset_index(drop=True) if not joint_states_df.empty else pd.DataFrame(columns=['time_ns', 'time_s'])

print('Expanded /joints columns:', list(joints_expanded.columns))
print('Expanded /encoders columns:', list(encoders_expanded.columns))
print('Expanded /joint_states columns:', list(joint_states_expanded.columns)[:20])

Expanded /joints columns: ['time_ns', 'time_s', 'joints_0', 'joints_1', 'joints_2', 'joints_3']
Expanded /encoders columns: ['time_ns', 'time_s', 'encoders_0', 'encoders_1']
Expanded /joint_states columns: ['time_ns', 'time_s', 'base_to_cylinder_joint', 'cylinder_to_arm1_joint', 'arm_link1_to_arm2_joint', 'arm_link2_to_arm3_joint', 'arm_link3_to_arm4_joint', 'arm_link4_to_armFork_joint', 'basket_link_to_armFork_joint', 'basket_crane_link_to_basket_link', 'winch_link_to_basket_crane_link']


In [5]:
def detect_command_events(telejoy_df):
    if telejoy_df.empty:
        return pd.DataFrame(columns=['command_index', 'command_sign', 'start_s', 'end_s', 'duration_s', 'mean_command'])

    events = []
    df = telejoy_df.sort_values('time_s').reset_index(drop=True)
    for command_index in range(COMMAND_VECTOR_LENGTH):
        col = f'cmd_{command_index}'
        if col not in df.columns:
            continue
        active = df[col].abs() > COMMAND_DEADBAND
        start_idx = None
        current_sign = 0
        for idx, is_active in enumerate(active):
            value = df.loc[idx, col]
            sign = int(np.sign(value)) if is_active else 0
            if is_active and start_idx is None:
                start_idx = idx
                current_sign = sign
            elif is_active and sign != current_sign:
                end_idx = idx - 1
                duration = df.loc[end_idx, 'time_s'] - df.loc[start_idx, 'time_s']
                if duration >= MIN_EVENT_DURATION_S:
                    segment = df.loc[start_idx:end_idx, col]
                    events.append({
                        'command_index': command_index,
                        'command_sign': current_sign,
                        'start_s': df.loc[start_idx, 'time_s'],
                        'end_s': df.loc[end_idx, 'time_s'],
                        'duration_s': duration,
                        'mean_command': float(segment.mean()),
                    })
                start_idx = idx
                current_sign = sign
            elif not is_active and start_idx is not None:
                end_idx = idx - 1
                duration = df.loc[end_idx, 'time_s'] - df.loc[start_idx, 'time_s']
                if duration >= MIN_EVENT_DURATION_S:
                    segment = df.loc[start_idx:end_idx, col]
                    events.append({
                        'command_index': command_index,
                        'command_sign': current_sign,
                        'start_s': df.loc[start_idx, 'time_s'],
                        'end_s': df.loc[end_idx, 'time_s'],
                        'duration_s': duration,
                        'mean_command': float(segment.mean()),
                    })
                start_idx = None
                current_sign = 0
        if start_idx is not None:
            end_idx = len(df) - 1
            duration = df.loc[end_idx, 'time_s'] - df.loc[start_idx, 'time_s']
            if duration >= MIN_EVENT_DURATION_S:
                segment = df.loc[start_idx:end_idx, col]
                events.append({
                    'command_index': command_index,
                    'command_sign': current_sign,
                    'start_s': df.loc[start_idx, 'time_s'],
                    'end_s': df.loc[end_idx, 'time_s'],
                    'duration_s': duration,
                    'mean_command': float(segment.mean()),
                })
    return pd.DataFrame(events).sort_values(['start_s', 'command_index']).reset_index(drop=True)

events_df = detect_command_events(telejoy_df)
events_df.to_csv(EVENTS_CSV, index=False)
display(events_df)
print(f'Detected command events: {len(events_df)}')

,command_index,command_sign,start_s,end_s,duration_s,mean_command


Detected command events: 0


In [6]:
def feedback_table(source):
    if source == 'joints':
        return joints_expanded
    if source == 'encoders':
        return encoders_expanded
    if source == 'joint_states':
        return joint_states_expanded
    raise ValueError(f'Unknown feedback source: {source}')

def feedback_column(action):
    source = action['feedback_source']
    index_or_name = action['feedback_index']
    if source in ('joints', 'encoders'):
        return f'{source}_{index_or_name}'
    return str(index_or_name)

def median_value_in_window(df, col, start_s, end_s):
    if df.empty or col not in df.columns:
        return np.nan
    window = df[(df['time_s'] >= start_s) & (df['time_s'] <= end_s)]
    if window.empty:
        return np.nan
    return float(window[col].median())

result_rows = []
for action in ACTIONS_TO_VALIDATE:
    action_events = events_df[events_df['command_index'] == action['command_index']] if not events_df.empty else pd.DataFrame()
    fb_df = feedback_table(action['feedback_source'])
    fb_col = feedback_column(action)
    expected_positive_sign = int(action['command_sign_to_feedback_sign'])

    for _, event in action_events.iterrows():
        before = median_value_in_window(fb_df, fb_col, event['start_s'] - PRE_WINDOW_S, event['start_s'])
        after = median_value_in_window(fb_df, fb_col, event['end_s'], event['end_s'] + POST_WINDOW_S)
        delta = after - before if np.isfinite(before) and np.isfinite(after) else np.nan
        measured_sign = int(np.sign(delta)) if np.isfinite(delta) and abs(delta) >= MIN_ABS_FEEDBACK_DELTA else 0
        expected_sign = int(event['command_sign']) * expected_positive_sign

        if measured_sign == 0:
            status = 'FAIL'
            reason = 'No measurable feedback movement in the expected response window.'
        elif measured_sign == expected_sign:
            status = 'PASS'
            reason = 'Measured feedback direction matches the expected command direction.'
        else:
            status = 'FAIL'
            reason = 'Measured feedback direction is opposite to the expected command direction.'

        result_rows.append({
            'action': action['action'],
            'command_index': int(action['command_index']),
            'command_sign': int(event['command_sign']),
            'feedback_source': action['feedback_source'],
            'feedback_signal': fb_col,
            'start_s': event['start_s'],
            'end_s': event['end_s'],
            'duration_s': event['duration_s'],
            'mean_command': event['mean_command'],
            'feedback_before': before,
            'feedback_after': after,
            'feedback_delta': delta,
            'expected_feedback_sign': expected_sign,
            'measured_feedback_sign': measured_sign,
            'status': status,
            'reason': reason,
        })

results_df = pd.DataFrame(result_rows)
results_df.to_csv(RESULTS_CSV, index=False)
display(results_df)

""


In [7]:
# Diagnostic helper: shows which /joints signal tends to change with each telejoy channel.
# Use this to confirm or refine ACTIONS_TO_VALIDATE. It is not the final KPI decision by itself.
diagnostic_rows = []
if not telejoy_df.empty and not joints_expanded.empty:
    telejoy_sorted = telejoy_df.sort_values('time_ns')
    joints_sorted = joints_expanded.sort_values('time_ns')
    merged = pd.merge_asof(telejoy_sorted, joints_sorted, on='time_ns', direction='nearest', tolerance=int(0.2e9), suffixes=('', '_fb'))
    for cmd_i in range(COMMAND_VECTOR_LENGTH):
        cmd_col = f'cmd_{cmd_i}'
        if cmd_col not in merged.columns:
            continue
        for fb_col in [c for c in merged.columns if c.startswith('joints_')]:
            valid = merged[[cmd_col, fb_col]].dropna()
            if len(valid) < 5:
                continue
            feedback_delta = valid[fb_col].diff()
            active = valid[cmd_col].abs() > COMMAND_DEADBAND
            if active.sum() < 3:
                continue
            corr = np.corrcoef(valid.loc[active, cmd_col], feedback_delta.loc[active].fillna(0.0))[0, 1]
            diagnostic_rows.append({'command_index': cmd_i, 'feedback_signal': fb_col, 'correlation_with_feedback_delta': corr})

diagnostic_df = pd.DataFrame(diagnostic_rows).sort_values('correlation_with_feedback_delta', key=lambda s: s.abs(), ascending=False) if diagnostic_rows else pd.DataFrame()
display(diagnostic_df.head(20))

""


In [8]:
summary_rows = []
if telejoy_df.empty:
    overall_status = 'INCONCLUSIVE'
    reason = 'No /telejoy command samples were received during acquisition.'
elif joints_df.empty and encoders_df.empty and joint_states_df.empty:
    overall_status = 'INCONCLUSIVE'
    reason = 'Command samples were received, but no feedback samples were available from /joints, /encoders, or /joint_states.'
elif events_df.empty:
    overall_status = 'INCONCLUSIVE'
    reason = f'No /telejoy command exceeded the deadband of {COMMAND_DEADBAND} for at least {MIN_EVENT_DURATION_S} s.'
elif results_df.empty:
    overall_status = 'INCONCLUSIVE'
    reason = 'Command events were detected, but none matched ACTIONS_TO_VALIDATE. Check the command_index mapping.'
elif (results_df['status'] == 'PASS').all():
    overall_status = 'PASS'
    reason = 'All configured command-to-motion direction checks passed.'
else:
    overall_status = 'FAIL'
    reason = 'At least one configured command-to-motion direction check failed.'

for source_name, df in [('/telejoy', telejoy_df), ('/joints', joints_df), ('/encoders', encoders_df), ('/joint_states', joint_states_df), ('/statusjoy', status_df)]:
    summary_rows.append({
        'metric': f'samples:{source_name}',
        'samples': int(len(df)),
        'status': 'AVAILABLE' if len(df) else 'MISSING',
        'reason': '',
    })

if not results_df.empty:
    for _, row in results_df.iterrows():
        summary_rows.append({
            'metric': f'direction:{row["action"]}',
            'samples': 1,
            'status': row['status'],
            'reason': row['reason'],
        })

summary_rows.append({
    'metric': 'TE02_004_overall',
    'samples': int(len(results_df)) if 'results_df' in globals() else 0,
    'status': overall_status,
    'reason': reason,
})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df)
print(f'Wrote summary: {SUMMARY_CSV}')
print(f'Overall TE02_004 status: {overall_status}')

,metric,samples,status,reason
0,samples:/telejoy,0,MISSING,
1,samples:/joints,1616,AVAILABLE,
2,samples:/encoders,27080,AVAILABLE,
3,samples:/joint_states,1616,AVAILABLE,
4,samples:/statusjoy,27079,AVAILABLE,
5,TE02_004_overall,0,INCONCLUSIVE,No /telejoy command samples were received duri...


Wrote summary: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004/te02_004_direction_summary.csv
Overall TE02_004 status: INCONCLUSIVE
